In [15]:
import anndata as ad
import scanpy as sc
import numpy as np
import pandas as pd


from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.inspection import permutation_importance
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

In [16]:
DATA_DIR = "/home/ubuntu/data/frangieh"
adata = sc.read_h5ad(f"{DATA_DIR}/rna_qc_filtered.h5ad")

In [ ]:
#Task 1 
#Can a classifier identify which treatment condition a cell came from using its gene expression profile?
#Which genes drive the separation?
#Features are gene expression values, target variable is treatment condition
#Random forest classifier

# train/test split FIRST, on raw (unnormalized) data
train_idx, test_idx = train_test_split(
    adata.obs_names,
    test_size=0.2,
    stratify=adata.obs["perturbation_2"],
    random_state=42
)

# split off a validation set from the training data, before any tuning happens
train_idx, val_idx = train_test_split(
    train_idx,
    test_size=0.2,
    stratify=adata.obs.loc[train_idx, "perturbation_2"],
    random_state=42
)

# calculate hvg on train split, using RAW counts (before normalization)
adata_train = adata[train_idx].copy()
sc.pp.highly_variable_genes(adata_train, n_top_genes=100, flavor="seurat_v3")
hvg_genes = adata_train.var_names[adata_train.var["highly_variable"]]

# normalize + log1p AFTER hvg selection, now that HVGs are already chosen
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

# train/val/test data splitting
X_train = adata[train_idx, hvg_genes].X
X_val = adata[val_idx, hvg_genes].X
X_test = adata[test_idx, hvg_genes].X
y_train = adata[train_idx].obs["perturbation_2"]
y_val = adata[val_idx].obs["perturbation_2"]
y_test = adata[test_idx].obs["perturbation_2"]

classifier = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    random_state=42,
    oob_score=True
)
classifier.fit(X_train, y_train)

In [ ]:
# Confirmation of hyperparameters on validation set
print("Out-of-Bag Score:", classifier.oob_score_)

y_val_pred = classifier.predict(X_val)
val_accuracy = accuracy_score(y_val, y_val_pred)
print("Validation Accuracy:", val_accuracy)
print(classification_report(y_val, y_val_pred))

In [ ]:
#Now we can evaluate model performance on the test set

print("Out-of-Bag Score:", classifier.oob_score_)

y_pred = classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

print(classification_report(y_test, y_pred))

In [ ]:
# raw confusion matrix as a numpy array
cm = confusion_matrix(y_test, y_pred, labels=classifier.classes_)
print(cm)

# plotted
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classifier.classes_)
disp.plot(cmap="Blues", xticks_rotation=45)
plt.title("Validation Confusion Matrix")
plt.show()

In [ ]:
#Extract feature importance to measure which genes drive the decision most

feature_importance = pd.DataFrame({
    "gene": hvg_genes,
    "importance": classifier.feature_importances_
})
feature_importance = feature_importance.sort_values(
    by="importance",
    ascending=False
)
print(feature_importance.head(20))

In [ ]:
per_importance_df = pd.DataFrame({
    "gene": list(hvg_genes),
    "importance": result.importances_mean,
    "std": result.importances_std
}).sort_values(by="importance", ascending=False)

print(per_importance_df.head(20))

In [ ]:
per_importance_df = pd.DataFrame({
    "gene": list(hvg_genes),
    "importance": result.importances_mean,
    "std": result.importances_std
}).sort_values(by="importance", ascending=False)

print(per_importance_df.head(20))

## Discussion
### Experimental setup
Data was split using the *train_test_split()* function of the skikit learn package. 80% of the cells were used for training and 20% for testing the model. To decrease model complexity and filter less important genes, the top 100 high-variance-genes were computed on the training dataset.
Hyperparameters of the model were adjusted based on early model performance. The initial model seemed to be overfitted (precision, recall and accuracy all close to 100%) so the model was constrained. *max_depth* ensures that the model can not split the data more than 8 times. *min_samples_split* forbids the model to split nodes with more than 10 samples, meaning the model could not split the data into leafs with only 1 feature each (which would require a *min_sample_split* of 2). *min_sample_leaf* further prevents such bahavior, as it forces all leaf notes to include at least 5 samples. <br>
*max_features* controls the amount of features the model can choose from at each split. The square root of the total number of features (in this case 10 as we looked at the 100 hvgs) is a common choice for random-forest-classifiers. As *oob_score* is set to True, the out-of-bag score, meaning an evaluation of the model using a fraction of the training data that each tree is not shown, is calculated for every tree.

### Model evaluation
An initial evaluation directly on the validation set suggested overfitting. After tuning the hyperparameters validation accuracy was 0.939, and final test accuracy was 0.938. The OOB-score was also taken into account for this. The model performs exeptionally well, as indicated by the very high accuracy, precision and recall. As robust measures against overfitting have been taken into account, it can be inferred that gene expression data is very suitable to predict the treatment group. <br>
The close agreement between the validation accuracy, test accuracy, and the out-of-bag score suggests that the model generalizes well to unseen data and is unlikely to have memorized the training set. Nevertheless, the results are based on a single train/test split, so evaluating the model with cross-validation or on an independent external dataset would provide a more robust estimate of its real-world performance. Overall, the consistently high performance indicates that the selected high-variance genes capture biologically meaningful differences between the treatment groups. Although the model achieved high accuracy, some uncertainty remains because its performance was evaluated on a single train-test split rather than multiple independent datasets.

### Biological interpretation
A closer look into feature importance and permutation importance reveals a select group of especially important genes. <br>
The top gene in both importance metrics, Indoleamine 2,3-dioxygenase 1 (IDO1) is an immune checkpoint regulator that can suppress T-cell activity, meaning regulation of this gene can be vital to immune evasion.
Top hits also include many human-leucocyte-antigen (HLA) genes. These are involved in presentation of internal proteins to CD8+ T-cells and play a vital role in the regular function of the immune system. <br>
Genes like HLA-DRA/HLA-DRB1 and CXCL9/10/11 are co-regulated as part of the same interferon-response programs.